# Module 02: Feature Engineering

Preprocessing and transformations: log-scaling Amount, dropping Time, outlier bounds, and stratified partitioning.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Load the data
try:
    data = pd.read_csv("../creditcard.csv")
    print("Loaded ../creditcard.csv successfully.")
except FileNotFoundError:
    data = pd.read_csv("creditcard.csv")
    print("Loaded creditcard.csv successfully.")

print(f"Original Shape: {data.shape}")

In [ ]:
# Drop 'Time' feature
data.drop(["Time"], axis=1, inplace=True)
print("Dropped 'Time' feature. Remaining columns:", data.shape[1])

In [ ]:
# Log-transform 'Amount' to reduce skewness
data["LogAmount"] = np.log1p(data["Amount"])
print("Created 'LogAmount' feature using np.log1p()")

In [ ]:
# Drop original 'Amount' column
data.drop(["Amount"], axis=1, inplace=True)
print("Dropped original 'Amount' column. Final columns:", data.shape[1])

In [ ]:
# Visualize the transformed feature
plt.figure(figsize=(7, 4))
sns.histplot(data["LogAmount"], bins=100, kde=True, color='#2b5c8f')
plt.title("Distribution of Log-Transformed Amount", fontsize=13, fontweight='bold')
plt.xlabel("Log(Amount + 1)")
plt.ylabel("Frequency")
plt.show()

## Comparing StandardScaler and RobustScaler

In [ ]:
# Compare StandardScaler vs RobustScaler on LogAmount
std_scaler = StandardScaler()
rob_scaler = RobustScaler()

scaled_std = std_scaler.fit_transform(data[['LogAmount']])
scaled_rob = rob_scaler.fit_transform(data[['LogAmount']])

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.histplot(scaled_std, bins=60, ax=axes[0], color='#28a745', kde=True)
axes[0].set_title('StandardScaler on LogAmount (Mean=0, Std=1)')

sns.histplot(scaled_rob, bins=60, ax=axes[1], color='#6f42c1', kde=True)
axes[1].set_title('RobustScaler on LogAmount (Median=0, IQR=1)')

plt.tight_layout()
plt.show()

## Outlier Boundary Analysis with IQR

In [ ]:
# Examine outliers in V14 for Fraud cases
v14_fraud = data['V14'].loc[data['Class'] == 1].values
q25, q75 = np.percentile(v14_fraud, 25), np.percentile(v14_fraud, 75)
v14_iqr = q75 - q25

v14_cut_off = v14_iqr * 1.5
v14_lower, v14_upper = q25 - v14_cut_off, q75 + v14_cut_off

print(f"V14 IQR: {v14_iqr:.3f}")
print(f"V14 Lower Bound: {v14_lower:.3f}, Upper Bound: {v14_upper:.3f}")
outliers = [x for x in v14_fraud if x < v14_lower or x > v14_upper]
print(f"Number of extreme V14 outliers in fraud cases: {len(outliers)}")

## Stratified Train-Test Split

In [ ]:
# Features (X) and Target (y)
X = data.drop("Class", axis=1)
y = data["Class"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Split Summary:")
print(f"X_train Shape: {X_train.shape}, y_train Class 1: {(y_train == 1).sum()} ({y_train.mean():.4%})")
print(f"X_test Shape:  {X_test.shape}, y_test Class 1:  {(y_test == 1).sum()} ({y_test.mean():.4%})")

## Summary of Feature Engineering

- Non-predictive Time column removed
- Highly skewed Amount normalized via log1p
- Scaling verified across numerical attributes
- Stratified 80/20 train/test partition established